In [ ]:
import gzip
import html
import os
from functools import lru_cache  # https://docs.python.org/zh-cn/3/library/functools.html

import ftfy # 字符修复的库
import regex as re # 正则表达式库


字符编码笔记：ASCII，Unicode 和 UTF-8 的区别： https://www.ruanyifeng.com/blog/2007/10/ascii_unicode_and_utf-8.html

In [ ]:
def bytes_to_unicode():
    """
    Returns list of utf-8 byte and a corresponding list of unicode strings.
    The reversible bpe codes work on unicode strings.
    This means you need a large # of unicode characters in your vocab if you want to avoid UNKs.
    When you're at something like a 10B token dataset you end up needing around 5K for decent coverage.
    This is a signficant percentage of your normal, say, 32K bpe vocab.
    To avoid that, we want lookup tables between utf-8 bytes and unicode strings.
    And avoids mapping to whitespace/control characters the bpe code barfs on.
    
    BPE（Byte Pair Encoding）是一种分词方法，它的输入和输出是基于 Unicode 字符串 的。为了更高效地处理大规模数据，我们需要一些机制来优化 BPE 的操作。
    Unicode 字符 是一种通用的字符编码，支持几乎所有语言，但字符种类特别多。
    如果直接用 BPE 编码所有 Unicode 字符，可能会导致词汇表大小过大，甚至出现未知字符（UNK）。
    为了避免这些问题，代码实现了一种查找表机制，将 UTF-8 单字节 和 Unicode 字符串 映射起来，简化处理。
    """
    # 从 ! (33) 到 ~ (126) 的字符。从 ¡ (161) 到 ¬ (172) 的字符。从 ® (174) 到 ÿ (255) 的字符。
    # ord 是获取unicode字符对应的码值， chr 根据码值获取对应字符
    # ascii-码中0-32是控制字符和空格，https://tool.oschina.net/commons?type=4
    bs = list(range(ord("!"), ord("~")+1))+list(range(ord("¡"), ord("¬")+1))+list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    # 对于不在bs范围内的数字[0-32,127-160,173],则使用256以后的unicode字符表示
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8+n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

print(bytes_to_unicode())

{33: '!', 34: '"', 35: '#', 36: '$', 37: '%', 38: '&', 39: "'", 40: '(', 41: ')', 42: '*', 43: '+', 44: ',', 45: '-', 46: '.', 47: '/', 48: '0', 49: '1', 50: '2', 51: '3', 52: '4', 53: '5', 54: '6', 55: '7', 56: '8', 57: '9', 58: ':', 59: ';', 60: '<', 61: '=', 62: '>', 63: '?', 64: '@', 65: 'A', 66: 'B', 67: 'C', 68: 'D', 69: 'E', 70: 'F', 71: 'G', 72: 'H', 73: 'I', 74: 'J', 75: 'K', 76: 'L', 77: 'M', 78: 'N', 79: 'O', 80: 'P', 81: 'Q', 82: 'R', 83: 'S', 84: 'T', 85: 'U', 86: 'V', 87: 'W', 88: 'X', 89: 'Y', 90: 'Z', 91: '[', 92: '\\', 93: ']', 94: '^', 95: '_', 96: '`', 97: 'a', 98: 'b', 99: 'c', 100: 'd', 101: 'e', 102: 'f', 103: 'g', 104: 'h', 105: 'i', 106: 'j', 107: 'k', 108: 'l', 109: 'm', 110: 'n', 111: 'o', 112: 'p', 113: 'q', 114: 'r', 115: 's', 116: 't', 117: 'u', 118: 'v', 119: 'w', 120: 'x', 121: 'y', 122: 'z', 123: '{', 124: '|', 125: '}', 126: '~', 161: '¡', 162: '¢', 163: '£', 164: '¤', 165: '¥', 166: '¦', 167: '§', 168: '¨', 169: '©', 170: 'ª', 171: '«', 172: '¬', 174: 

In [ ]:
def get_pairs(word):
    """Return set of symbol pairs in a word.
    Word is represented as tuple of symbols (symbols being variable-length strings).
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs


print(get_pairs("hello world!"))
# 由于pairs是set，所以输出是乱序的，并不是按照{('h', 'e'), ('e', 'l'),...('d', '!')}

{('e', 'l'), ('w', 'o'), ('l', 'd'), ('h', 'e'), ('l', 'l'), (' ', 'w'), ('r', 'l'), ('d', '!'), ('o', 'r'), ('o', ' '), ('l', 'o')}


In [ ]:
def basic_clean(text):
    # 使用 ftfy 库修复文本中的错误编码问题
    text = ftfy.fix_text(text)
    # 使用 html.unescape 将 HTML 实体（如 &amp;）转义为普通字符（如 &）
    text = html.unescape(html.unescape(text))
    # 移除文本开头和结尾的多余空格
    return text.strip()

text = "â€œHello &amp; Welcome to the World!â€"
print(basic_clean(text))

text = "   &#34;Python &lt;3 is Awesome!&#34;   "
print(basic_clean(text))

"Hello & Welcome to the World!"
"Python <3 is Awesome!"


In [ ]:
def whitespace_clean(text):
    text = re.sub(r'\s+', ' ', text)  # 清楚多余1个的空格
    text = text.strip()
    return text

text = "   h ello    world   \n python "
print(whitespace_clean(text))


h ello world python


## BPE介绍：
参考： https://zhuanlan.zhihu.com/p/424631681 或者 直接询问gpt，解释的简单清晰。

BPE 的主要目标是将文本分割成一个较小的子词单元集合，既避免单字分词的过细（导致序列过长），又避免整词分词的过粗（导致大量未知词）。
基本步骤：

**1初始化**：

将文本中的每个单词分解为一个个字符序列，并将字符序列作为初始词汇表。例如，单词 hello 可能会被分解为 ["h", "e", "l", "l", "o"]。

**2统计字符对频率**：

统计所有出现的字符对（相邻的两个字符）的频率。例如，("l", "l") 的频率可能高于其他字符对。

**3合并最频繁的字符对**：

将频率最高的字符对合并为一个新的符号，例如将 ("l", "l") 合并为 "ll"。

**4更新语料**：

使用新的符号替换所有出现的该字符对。重复步骤 2 和 3，直到达到预定义的词汇表大小或不再有字符对可合并。
生成词汇表：

将最终的字符和子词作为模型的词汇表。



In [ ]:
import os
@lru_cache()
def default_bpe():
    return os.path.join(os.getcwd(), "bpe_simple_vocab_16e6.txt.gz")


class SimpleTokenizer(object):
    def __init__(self, bpe_path: str = default_bpe()):
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        merges = gzip.open(bpe_path).read().decode("utf-8").split('\n')
        merges = merges[1:49152-256-2+1]
        merges = [tuple(merge.split()) for merge in merges]
        vocab = list(bytes_to_unicode().values())
        vocab = vocab + [v+'</w>' for v in vocab]
        for merge in merges:
            vocab.append(''.join(merge))
        vocab.extend(['<|startoftext|>', '<|endoftext|>'])
        print("vocab len:{}".format(len(vocab)))
        self.encoder = dict(zip(vocab, range(len(vocab))))
        self.decoder = {v: k for k, v in self.encoder.items()}
        self.bpe_ranks = dict(zip(merges, range(len(merges))))
        self.cache = {'<|startoftext|>': '<|startoftext|>', '<|endoftext|>': '<|endoftext|>'}
        self.pat = re.compile(r"""<\|startoftext\|>|<\|endoftext\|>|'s|'t|'re|'ve|'m|'ll|'d|[\p{L}]+|[\p{N}]|[^\s\p{L}\p{N}]+""", re.IGNORECASE)

    def bpe(self, token):
        if token in self.cache:
            return self.cache[token]
        word = tuple(token[:-1]) + ( token[-1] + '</w>',)
        pairs = get_pairs(word)

        if not pairs:
            return token+'</w>'
        step = 1
        while True:
            bigram = min(pairs, key = lambda pair: self.bpe_ranks.get(pair, float('inf')))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word)-1 and word[i+1] == second:
                    new_word.append(first+second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            print(f"step:{step}:", new_word)
            step += 1
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = ' '.join(word)
        self.cache[token] = word
        return word

    def encode(self, text):
        bpe_tokens = []
        text = whitespace_clean(basic_clean(text)).lower()
        for token in re.findall(self.pat, text):
            print(f"process word: {token}")
            token = ''.join(self.byte_encoder[b] for b in token.encode('utf-8'))
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(' '))
        return bpe_tokens

    def decode(self, tokens):
        text = ''.join([self.decoder[token] for token in tokens])
        text = bytearray([self.byte_decoder[c] for c in text]).decode('utf-8', errors="replace").replace('</w>', ' ')
        return text


In [ ]:
tokenizer = SimpleTokenizer()
text = "hello tokenizer"
bpe_tokens = tokenizer.encode(text)
print(bpe_tokens)

vocab len:49408
process word: hello
step:1: ('h', 'el', 'l', 'o</w>')
step:2: ('h', 'ell', 'o</w>')
step:3: ('h', 'ello</w>')
step:4: ('hello</w>',)
process word: tokernizer
step:1: ('t', 'o', 'k', 'er', 'n', 'i', 'z', 'e', 'r</w>')
step:2: ('t', 'o', 'k', 'er', 'n', 'i', 'z', 'er</w>')
step:3: ('to', 'k', 'er', 'n', 'i', 'z', 'er</w>')
step:4: ('to', 'k', 'er', 'ni', 'z', 'er</w>')
step:5: ('to', 'ker', 'ni', 'z', 'er</w>')
step:6: ('to', 'ker', 'ni', 'zer</w>')
[3306, 580, 2352, 697, 3716]
